In [26]:
from google.cloud import bigquery
import os
import plotly.express as px
import pandas as pd
from pyproj import Transformer


# Option A : Via variable d'environnement (Recommandé) 
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../data/auth/sante-et-territoires-216daad01024.json" 
client = bigquery.Client() 
# Option B : Directement dans le code# 
#client = bigquery.Client.from_service_account_json("C:\\Users\stefl\Documents\Projet\sante-et-territoires-216daad01024.json")
#export GOOGLE_APPLICATION_CREDENTIALS="C:\Users\stefl\Documents\Projet\sante-et-territoires-216daad01024.json"

#client = bigquery.Client()

query = """
SELECT *
FROM `sante-et-territoires.finess.finess_etablissements_clean`
"""

df = client.query(query).to_dataframe()




In [27]:
df_communes = pd.read_csv("../data/geo/communes-france-2025.csv", sep=",", encoding="utf-8")

/tmp/ipykernel_7305/1037513388.py:1: DtypeWarning:

Columns (1,9,11,13,17) have mixed types. Specify dtype option on import or set low_memory=False.



In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102553 entries, 0 to 102552
Data columns (total 23 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   nofinesset         102553 non-null  object
 1   nofinessej         102553 non-null  object
 2   rs                 102553 non-null  object
 3   rslongue           81834 non-null   object
 4   complrs            9301 non-null    object
 5   numvoie            86798 non-null   object
 6   typvoie            97606 non-null   object
 7   voie               100577 non-null  object
 8   compvoie           3971 non-null    object
 9   compldistrib       11902 non-null   object
 10  lieuditbp          10936 non-null   object
 11  commune            102553 non-null  object
 12  ligneacheminement  102553 non-null  object
 13  departement        102553 non-null  object
 14  libdepartement     102553 non-null  object
 15  categetab          102553 non-null  object
 16  libcategetab       1

Changement de coordonnées pour avoir des latitudes longitudes 


In [29]:




# Création du transformateur Lambert 93 -> WGS84
transformer = Transformer.from_crs("EPSG:2154", "EPSG:4326", always_xy=True)

# Conversion
df["longitude"], df["latitude"] = transformer.transform(
    df["coordxet"].values,
    df["coordyet"].values
)

# Aperçu
print(df[["nofinesset", "coordxet", "coordyet", "latitude", "longitude"]].head())

  nofinesset   coordxet   coordyet   latitude  longitude
0  060024411  1041862.7  6297002.1  43.689403   7.241235
1  060785003  1044997.0  6301126.4  43.724939   7.282816
2  060788957  1044172.6  6300626.5  43.720849   7.272267
3  060010899  1044798.6  6301088.3  43.724694   7.280333
4  060788262  1041871.1  6297745.7  43.696081   7.241834


In [30]:
df['libcategetab'].value_counts()

libcategetab
Pharmacie d'Officine                                            20084
Service autonomie aide (SAA)                                    10103
Etablissement d'hébergement pour personnes âgées dépendantes     7402
Laboratoire de Biologie Médicale                                 4527
Centre de Santé                                                  3379
                                                                ...  
Service de Travailleuses Familiales                                 2
Service de Repas à Domicile                                         2
Etablissement de Soins du Service de Santé des Armées               1
Ecole des Hautes Etudes en Santé Publique (E.H.E.S.P.)              1
Service d'Aide aux Personnes Agées                                  1
Name: count, Length: 172, dtype: int64

In [31]:
# Regroupement des catégories de types d'établissements
def regrouper_categorie(cat):
    cat = cat.lower()

    if any(x in cat for x in ["hospital", "clinique", "soins", "dialyse", "ssr", "had", "cancer"]):
        return "Hopitaux cliniques"

    if any(x in cat for x in ["handicap", "ime", "itep", "mas", "fam", "esat", "sensoriel", "e.s.a.t.", "i.m.e.", "i.t.e.p.", "m.a.s.", "s.a.v.s.", "foyer de vie", "entreprise adaptée", "esat", "institut d'éducation motrice", "service mandataire judiciaire à la protection des majeurs", "c.a.m.s.p."]):
        return "Médico-social handicap"

    if any(x in cat for x in ["ehpad", "ehpa", "autonomie", "personnes âgées", "longue durée", "personnes agées"]):
        return "Personnes âgées"

    if any(x in cat for x in ["chrs", "casa", "c.a.d.a", "hébergement", "foyer", "maison relais"]):
        return "Social / Hébergement"

    if any(x in cat for x in ["pharmacie", "centre de santé", "maison de santé", "laboratoire", "c.m.p.", "cabinet", "infirmier", "kinésithérapeute", "dentiste", "opticien", "structure dispensatrice à domicile d'oxygène à usage médical", "maison médicale de garde (MMG)"]):
        return "Soins de ville"

    if any(x in cat for x in ['prévention', "vaccination", "dépistage", "csapa", "caarud" ]):
        return "Prévention / Santé publique"

    if any(x in cat for x in ["enfance", "camsp", "aemo", "aed", "pouponnière", "maison d'enfants", "educative", "protection maternelle et infantile", "protection infantile", "pmi", "p.m.i."]):
        return "Enfance / Protection"

    if any(x in cat for x in ["cpts", "mdph", "coordination", "groupement"]):
        return "Coordination / Administration"
    if any(x in cat for x in ["centre d'accueil", "accueil"]):
            return "Centres d'accueil"
    if any(x in cat for x in ["ecoles"]):
        return "Ecoles"
    return "Autres"

# Application
df["groupe"] = df["libcategetab"].apply(regrouper_categorie)

In [32]:
df['groupe'].value_counts()

groupe
Soins de ville                   35294
Personnes âgées                  20313
Médico-social handicap           16239
Hopitaux cliniques               12014
Enfance / Protection              4969
Centres d'accueil                 3421
Autres                            2675
Coordination / Administration     2586
Social / Hébergement              2564
Ecoles                            1372
Prévention / Santé publique       1106
Name: count, dtype: int64

In [33]:
#nettoyage des doublons
df_clean = df.drop_duplicates()

In [34]:
#je crée un colonne code_insee à partir de la colonne code commune et de la colonne departement
df_clean["code_insee"] = df_clean["departement"].astype(str) + df_clean["commune"].astype(str)
df_clean.head()

,nofinesset,nofinessej,rs,rslongue,complrs,numvoie,typvoie,voie,compvoie,compldistrib,...,siret,codeape,coordxet,coordyet,dateouv,dateautor,longitude,latitude,groupe,code_insee
0,060024411,060785011,CHU DE NICE ANTENNE SMUR SITE HPNCL,CHU NICE ANTENNE SMUR SITE HOP PEDIATRIQUES NI...,None,57,AV,DE LA CALIFORNIE,None,None,...,26060070500099,8610Z,1041862.7,6297002.1,2007-03-28,2007-03-28,7.241235,43.689403,Hopitaux cliniques,06088
1,060785003,060785011,CHU DE NICE HOPITAL PASTEUR,CHU DE NICE HOPITAL PASTEUR,None,30,AV,DE LA VOIE ROMAINE,None,None,...,26060070500099,8610Z,1044997.0,6301126.4,1980-01-01,1980-01-01,7.282816,43.724939,Hopitaux cliniques,06088
2,060788957,060785011,CHU DE NICE HOPITAL DE CIMIEZ,CHU DE NICE HOPITAL DE CIMIEZ,None,4,AV,REINE VICTORIA,None,None,...,26060070500040,8610Z,1044172.6,6300626.5,1904-04-04,1904-04-04,7.272267,43.720849,Hopitaux cliniques,06088
3,060010899,060785011,INSTITUT UNIVERSITAIRE FACE ET COU,IUFC INSTITUT UNIVERSITAIRE DE LA FACE ET DU COU,None,31,AV,DE VALOMBROSE,None,None,...,26060070500081,8610Z,1044798.6,6301088.3,2014-10-28,2014-10-28,7.280333,43.724694,Hopitaux cliniques,06088
4,060788262,060785011,HJ CHU NICE,HJ CHU NICE,None,35,BD,DE LA MADELEINE,None,None,...,26060070500040,8899B,1041871.1,6297745.7,1978-03-01,1978-03-01,7.241834,43.696081,Hopitaux cliniques,06088


In [35]:
df_clean = df_clean.rename(columns={
    "rs": "raison_sociale",
    "libcategetab": "categorie",
    "commune": "code_commune",
    "nofinessej": "numero finess etablissement juridique",
      'nofinesset': "numero_finess_etablissement", 
        'libdepartement': "libelle_departement",
         'groupe' : "type d etablissements",
})

In [36]:
#je filtre les données sur les départements de la région Occitanie
departements_occitanie = ["09","11", "12", "30","31", "32", "34", "46", "48","65", "66", "81", "82",  ]
df_occitanie = df_clean[df_clean["departement"].isin(departements_occitanie)]

In [37]:
# latitudes longitudes en float
df_occitanie.loc[:,'latitude'] = df['latitude'].astype(float)
df_occitanie.loc[:,'longitude'] = df['longitude'].astype(float)

In [38]:
#création d'une carte des Hopitaux cliniques
import plotly.express as px

# 1. Filtrer sur un groupe
df_filtre = df_occitanie[df_occitanie['type d etablissements'] == "Hopitaux cliniques"]




# 2. Créer la carte (nouvelle API)
fig = px.scatter_map(
    df_filtre,
    lat="latitude",
    lon="longitude",
    hover_name="raison_sociale",
    hover_data={
        "latitude": False,
        "longitude": False
    },
    color="type d etablissements",
    zoom=5,
    height=700
)

# 3. Style de carte (plus de mapbox_style)
fig.update_layout(
    map_style="open-street-map",   # nouvelle syntaxe
    margin={"r":0, "t":0, "l":0, "b":0}
)

fig.show()

In [39]:
#j'exporte en csv dans processed la partie occitanie

df_occitanie.to_csv("../data/processed/etablissements_occitanie.csv", index=False, sep=";", encoding="utf-8")


In [40]:
#j'exporte en csv dans processed l occitanie plus les départements limitrophes
departement_limitrophes_et_occitanie= ["64", "40", "47", "24", "19", "15", "43", "07", "84", "13", "09","11", "12", "30","31", "32", "34", "46", "48","65", "66", "81", "82"]
df_limit= df_clean[df_clean["departement"].isin(departement_limitrophes_et_occitanie)]
df_limit.info()
df_limit.to_csv("../data/processed/etablissements_limit.csv", index=False, sep=";", encoding="utf-8")

<class 'pandas.core.frame.DataFrame'>
Index: 19090 entries, 10 to 102511
Data columns (total 27 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   numero_finess_etablissement            19090 non-null  object 
 1   numero finess etablissement juridique  19090 non-null  object 
 2   raison_sociale                         19090 non-null  object 
 3   rslongue                               16822 non-null  object 
 4   complrs                                1578 non-null   object 
 5   numvoie                                14975 non-null  object 
 6   typvoie                                17948 non-null  object 
 7   voie                                   18551 non-null  object 
 8   compvoie                               552 non-null    object 
 9   compldistrib                           2626 non-null   object 
 10  lieuditbp                              1984 non-null   object 
 11  code_

NameError: name 'df_communes_occitanie' is not defined